In [ ]:


import re 
from pathlib import Path 

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as functional
from transformers import AutoModel, AutoTokenizer



In [ ]:

MODEL_NAME = "distilbert-base-uncased"

# Transparent hand-set weights. These are not learned from popularity labels.
WEIGHTS = {
    "urgency": 0.20, # 
    "emotion": 0.15,
    "title_length": 0.15,
    "clarity": 0.10,
    "word_variety": 0.10,
    "text_length": 0.10,
    "transformer_match": 0.15,
    "batch_novelty": 0.05,
}

URGENCY_WORDS = {
    "breaking", "urgent", "alert", "emergency", "warning", "today",
    "live", "just", "now", "latest", "developing", "exclusive",
}
EMOTION_WORDS = {
    "crisis", "shocking", "dramatic", "fear", "deadly", "danger",
    "dangerous", "massive", "major", "historic", "surge", "threat",
    "crash", "attack", "victory",
}
WORD_RE = re.compile(r"[A-Za-z]+(?:['’-][A-Za-z]+)?") # Regular expression for matching words 


In [7]:


class NewsEncoder:
    """Pretrained DistilBERT text encoder that runs on CPU."""

    def __init__(self, model_name=MODEL_NAME):
        self.model_name = model_name
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.to("cpu")
        self.model.eval()

    @torch.inference_mode()
    def encode(self, texts, batch_size=8):
        """Return one normalized transformer embedding for each text."""
        if len(texts) == 0:
            return np.empty((0, 0), dtype=np.float32)

        output_vectors = []
        for start in range(0, len(texts), batch_size):
            batch = [str(text or "") for text in texts[start:start + batch_size]]
            tokens = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt",
            )
            hidden = self.model(**tokens).last_hidden_state
            mask = tokens["attention_mask"].unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            pooled = functional.normalize(pooled, p=2, dim=1)
            output_vectors.append(pooled.cpu().numpy().astype(np.float32))

        return np.concatenate(output_vectors, axis=0)



In [8]:

def clean_news_data(frame):
    """Keep just title and description, normalize whitespace, remove blanks."""
    frame = frame.copy()
    frame.columns = [str(name).strip().lower() for name in frame.columns]
    required = {"title", "description"}
    if not required.issubset(frame.columns):
        raise ValueError("CSV must contain 'title' and 'description' columns.")

    frame = frame[["title", "description"]].copy()
    for column in ["title", "description"]:
        frame[column] = (
            frame[column].fillna("").astype(str)
            .str.replace(r"\s+", " ", regex=True).str.strip()
        )
    frame = frame[(frame.title != "") | (frame.description != "")]
    return frame.drop_duplicates().reset_index(drop=True)



In [9]:

def merge_news_csvs(input_folder="data/raw", output_file="data/processed/news.csv"):
    """Merge all CSV files in a folder and save cleaned title/description."""
    files = sorted(Path(input_folder).glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"No CSV files found in {input_folder}")

    cleaned_frames = [clean_news_data(pd.read_csv(file)) for file in files]
    merged = clean_news_data(pd.concat(cleaned_frames, ignore_index=True))
    output_path = Path(output_file)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(output_path, index=False)
    return merged




In [10]:
def _words(text):
    return [word.lower() for word in WORD_RE.findall(str(text or ""))]


def _clip(value):
    return float(np.clip(value, 0, 1))


def _cosine(left, right):
    denominator = float(np.linalg.norm(left) * np.linalg.norm(right))
    if denominator == 0:
        return 0.0
    return float(np.dot(left, right) / denominator)



In [11]:

def _get_signals(title, description, title_vector, description_vector, novelty):
    title_words = _words(title)
    words = _words(title + " " + description)
    word_count = len(words)

    urgency_count = sum(word in URGENCY_WORDS for word in words)
    emotion_count = sum(word in EMOTION_WORDS for word in words)
    average_word_length = sum(map(len, words)) / max(word_count, 1)
    transformer_match = (_cosine(title_vector, description_vector) + 1) / 2

    return {
        "urgency": _clip(urgency_count / 2),
        "emotion": _clip(emotion_count / 3),
        "title_length": _clip(1 - abs(len(title_words) - 10) / 12),
        "clarity": _clip(1 - abs(average_word_length - 5) / 5),
        "word_variety": _clip(len(set(words)) / max(word_count, 1)),
        "text_length": _clip(1 - abs(word_count - 45) / 65),
        "transformer_match": _clip(transformer_match),
        "batch_novelty": _clip(novelty),
    }



In [12]:

def _make_result(title, description, signals, model_name):
    score = round(100 * sum(WEIGHTS[name] * signals[name] for name in WEIGHTS), 1)
    labels = {
        "urgency": "Urgency wording",
        "emotion": "Emotional language",
        "title_length": "Title length",
        "clarity": "Word clarity proxy",
        "word_variety": "Lexical variety",
        "text_length": "Text length fit",
        "transformer_match": "Transformer title-description match",
        "batch_novelty": "Novelty within this batch",
    }
    ordered = sorted(signals.items(), key=lambda item: item[1], reverse=True)
    highlights = [
        f"{labels[name]} signal: {value:.0%}."
        for name, value in ordered[:3] if value >= 0.55
    ]
    if not highlights:
        highlights = ["No single content signal stands out strongly."]
    lower = [labels[name] for name, value in ordered[-3:] if value < 0.45]
    if lower:
        highlights.append("Lower proxy signals: " + ", ".join(lower) + ".")

    return {
        "title": title,
        "description": description,
        "score": score,
        "signals": signals,
        "highlights": highlights,
        "model_name": model_name,
    }



In [13]:

def score_article(title, description, encoder):
    """Return a proxy score and explanation for one title/description pair."""
    vectors = encoder.encode([title, description])
    signals = _get_signals(title, description, vectors[0], vectors[1], 0.5)
    return _make_result(title, description, signals, encoder.model_name)



In [14]:

def rank_articles(frame, encoder):
    """Score and rank a DataFrame with title and description columns."""
    frame = clean_news_data(frame)
    if frame.empty:
        return pd.DataFrame(columns=["rank", "score", "title", "description", "highlights"])

    titles = frame.title.tolist()
    descriptions = frame.description.tolist()
    combined = [title + ". " + description for title, description in zip(titles, descriptions)]
    count = len(frame)
    vectors = encoder.encode(titles + descriptions + combined)

    nearest_similarity = []
    for index in range(count):
        if count == 1:
            nearest_similarity.append(0.0)
        else:
            peers = [i for i in range(count) if i != index]
            similarity = [
                _cosine(vectors[2 * count + index], vectors[2 * count + peer])
                for peer in peers
            ]
            nearest_similarity.append(max(similarity))

    novelty = [0.5] * count
    if count > 1:
        if max(nearest_similarity) - min(nearest_similarity) > 1e-8:
            ranks = np.argsort(np.argsort(nearest_similarity, kind="stable"), kind="stable")
            novelty = [1 - float(rank) / (count - 1) for rank in ranks]

    results = []
    for index, (title, description) in enumerate(zip(titles, descriptions)):
        signals = _get_signals(
            title, description, vectors[index], vectors[count + index], novelty[index]
        )
        result = _make_result(title, description, signals, encoder.model_name)
        row = {
            "score": result["score"],
            "title": title,
            "description": description,
            "highlights": " ".join(result["highlights"]),
        }
        row.update({name + "_signal": value for name, value in signals.items()})
        results.append(row)

    ranked = pd.DataFrame(results).sort_values("score", ascending=False).reset_index(drop=True)
    ranked.insert(0, "rank", range(1, len(ranked) + 1))
    return ranked
